# Detector Training Controller

Runs reproducible training experiments across YOLOX-Tiny, SSDLite320 MobileNetV3-Large, and Faster R-CNN ResNet50-FPN. Designed to support independent per-GPU runs on Kaggle T4 x2 and checkpoint resumption.

In [ ]:
from pathlib import Path

REPO_ROOT = Path('/kaggle/working/EdgeVisionInspector')
RUNS_DIR = REPO_ROOT / 'runs'
RUNS_DIR.mkdir(parents=True, exist_ok=True)

wrapper = RUNS_DIR / '_run_one.py'
wrapper.write_text('''
import json
import sys
from pathlib import Path

REPO_ROOT = Path('/kaggle/working/EdgeVisionInspector')
sys.path.insert(0, str(REPO_ROOT / 'src'))

from edgevision.training.train import train_detector

EXPERIMENTS = [
    ('configs/yolox_tiny.yaml',   [320, 320], 'runs/yolox_tiny_320'),
    ('configs/ssdlite.yaml',      [320, 320], 'runs/ssdlite_320'),
    ('configs/faster_rcnn.yaml',  [320, 320], 'runs/faster_rcnn_320'),
    ('configs/yolox_tiny.yaml',   [224, 224], 'runs/yolox_tiny_224'),
    ('configs/faster_rcnn.yaml',  [224, 224], 'runs/faster_rcnn_224'),
]

idx = int(sys.argv[1])
cfg_rel, input_size, out_rel = EXPERIMENTS[idx]
out_dir = REPO_ROOT / out_rel
last_ckpt = out_dir / 'last.pt'

summary = train_detector(
    model_cfg_path=REPO_ROOT / cfg_rel,
    dataset_cfg_path=REPO_ROOT / 'configs/dataset.yaml',
    preprocess_cfg_path=REPO_ROOT / 'configs/preprocess.yaml',
    train_manifest=REPO_ROOT / 'data/manifests/train.jsonl',
    val_manifest=REPO_ROOT / 'data/manifests/val.jsonl',
    output_dir=out_dir,
    device_str='cuda:0',
    input_size_override=tuple(input_size),
    resume_from=last_ckpt if last_ckpt.exists() else None,
)
print(json.dumps(summary, default=str))
''', encoding='utf-8')

print(f"Wrapper written to {wrapper}")

In [ ]:
import os
import subprocess
import sys
import threading
from pathlib import Path

REPO_ROOT = Path('/kaggle/working/EdgeVisionInspector')
RUNS_DIR = REPO_ROOT / 'runs'
WRAPPER = RUNS_DIR / '_run_one.py'
print_lock = threading.Lock()

def launch(run_index, gpu_id):
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = str(gpu_id)
    env['EDGEVISION_ROOT'] = str(REPO_ROOT)
    env['PYTHONUNBUFFERED'] = '1'
    log_path = RUNS_DIR / f'_run_{run_index}.log'
    proc = subprocess.Popen(
        [sys.executable, str(WRAPPER), str(run_index)],
        cwd=str(REPO_ROOT),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    return proc, log_path, run_index

def stream(proc, log_path, run_index):
    try:
        with open(log_path, 'w', encoding='utf-8') as log_f:
            if proc.stdout is not None:
                for line in proc.stdout:
                    log_f.write(line)
                    log_f.flush()
                    with print_lock:
                        print(f'[run{run_index}] {line.rstrip()}', flush=True)
    finally:
        rc = proc.wait()
        with print_lock:
            print(f'[run{run_index}] EXIT CODE {rc}', flush=True)

# ---- choose the pair for this batch ----
BATCH = [(0, 0), (1, 1)]   # (run_index, gpu_id) — YOLOX 320 on GPU 0, SSDLite 320 on GPU 1

jobs = [launch(ri, gi) for ri, gi in BATCH]
threads = [threading.Thread(target=stream, args=j, daemon=True) for j in jobs]
[t.start() for t in threads]
[t.join() for t in threads]

print(f"Batch complete. Exit codes: {[j[0].returncode for j in jobs]}")